# ETF OOS Pipeline v3 — 개선판
- `config.yaml` 한 곳에서 모든 설정 관리
- 추가 기술지표: RSI, MACD, Bollinger Band, ATR, 52주 고저
- 추가 모델: XGBoost, LightGBM (+ 클래스 불균형 처리)
- **2단계 Optuna**: Stage1(타깃 탐색) → Stage2(모델·하이퍼파라미터 탐색)
- 주요 결과 CSV 저장 + 실험 비교 summary CSV 누적

In [1]:
import yaml, warnings, os, json, gc, pickle
from pathlib import Path
warnings.filterwarnings("ignore")

with open("config.yaml", "r") as f:
    CFG = yaml.safe_load(f)

EXPERIMENT_NAME    = CFG["experiment"]["name"]
ETF_CODE           = CFG["etf"]["code"]
START_DATE         = CFG["etf"]["start_date"]
END_DATE           = CFG["etf"].get("end_date")
BASE_DATE          = CFG["base_date"]

OPTUNA_VALID_MONTHS = CFG["periods"]["optuna_valid_months"]
SIM_TEST_MONTHS     = CFG["periods"]["sim_test_months"]

N_DAYS_CANDIDATES               = CFG["target"]["n_days_candidates"]
TARGET_RETURN_THRESHOLD_CANDIDATES = CFG["target"]["return_threshold_candidates"]

VIF_THRESHOLD       = CFG["features"]["vif_threshold"]
FEATURE_SELECT_YEARS= CFG["features"]["feature_select_years"]
LAG_DAYS            = CFG["features"]["lag_days"]
TOP_N_MAX           = CFG["features"]["top_n_max"]

USE_RSI      = CFG["features"]["use_rsi"]
RSI_PERIOD   = CFG["features"]["rsi_period"]
USE_MACD     = CFG["features"]["use_macd"]
MACD_FAST    = CFG["features"]["macd_fast"]
MACD_SLOW    = CFG["features"]["macd_slow"]
MACD_SIG     = CFG["features"]["macd_signal"]
USE_BB       = CFG["features"]["use_bollinger"]
BB_PERIOD    = CFG["features"]["bollinger_period"]
BB_STD       = CFG["features"]["bollinger_std"]
USE_ATR      = CFG["features"]["use_atr"]
ATR_PERIOD   = CFG["features"]["atr_period"]
USE_52W      = CFG["features"]["use_52w"]

EXTERNAL_TICKERS      = CFG["external_tickers"]
EXTERNAL_FEATURE_TYPES= CFG["external_feature_types"]

RANDOM_STATE = CFG["importance"]["random_state"]
N_RF_RUNS    = CFG["importance"]["n_rf_runs"]
N_REPEATS    = CFG["importance"]["n_repeats"]

S1_OBJ_METRIC   = CFG["stage1"]["objective_metric"]
S1_MIN_EVAL     = CFG["stage1"]["min_valid_eval_count"]
S1_MIN_PRED1    = CFG["stage1"]["min_valid_pred_1_count"]
S1_MODEL        = CFG["stage1"]["model_name"]
S1_N_EST        = CFG["stage1"]["n_estimators"]
S1_TOP_N        = CFG["stage1"]["top_n_fixed"]
S1_N_SEEDS      = CFG["stage1"]["n_seeds"]

S2_N_TRIALS     = CFG["optuna_stage2"]["n_trials"]
S2_OBJ_METRIC   = CFG["optuna_stage2"]["objective_metric"]
S2_MIN_EVAL     = CFG["optuna_stage2"]["min_valid_eval_count"]
S2_MIN_PRED1    = CFG["optuna_stage2"]["min_valid_pred_1_count"]
S2_MODELS       = CFG["optuna_stage2"]["model_candidates"]
S2_TOP_N_RANGE  = CFG["optuna_stage2"]["top_n_range"]
S2_THRESH_RANGE = CFG["optuna_stage2"]["pred_threshold_range"]
S2_THRESH_STEP  = CFG["optuna_stage2"]["pred_threshold_step"]

INITIAL_CASH    = CFG["simulation"]["initial_cash"]
BUY_RATIO       = CFG["simulation"]["buy_ratio"]
MIN_CASH_RATIO  = CFG["simulation"]["min_cash_ratio"]

LOAD_FROM_CACHE = CFG["cache"]["load_from_cache"]

_cache_tag = f"{ETF_CODE}_{BASE_DATE.replace('-','')}"
CACHE_DIR  = Path(f"cache_{_cache_tag}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path(f"experiments_v3/{EXPERIMENT_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_cache(name, obj):
    path = CACHE_DIR / f"{name}.pkl"
    with open(path, "wb") as f: pickle.dump(obj, f)
    print(f"[CACHE SAVED] {path}")

def load_cache(name):
    path = CACHE_DIR / f"{name}.pkl"
    with open(path, "rb") as f: obj = pickle.load(f)
    print(f"[CACHE LOADED] {path}")
    return obj

print(f"Experiment : {EXPERIMENT_NAME}")
print(f"ETF        : {ETF_CODE}  BASE_DATE={BASE_DATE}")
print(f"Output dir : {OUTPUT_DIR}")


Experiment : v3_baseline
ETF        : SMH  BASE_DATE=2026-05-31
Output dir : experiments_v3/v3_baseline


In [2]:
import numpy as np
import pandas as pd
import FinanceDataReader as fdr
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss, confusion_matrix
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from statsmodels.stats.outliers_influence import variance_inflation_factor

import xgboost as xgb
import lightgbm as lgb
import plotly.graph_objects as go
from IPython.display import display

def display_df(df, n=20):
    display(df.head(n))

# ── 기술지표 헬퍼 ─────────────────────────────────────────────────────────────
def _rsi(close, period=14):
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

def _atr(high, low, close, period=14):
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

# ── 1. 데이터 로드 ────────────────────────────────────────────────────────────
def load_price_data(ticker, start_date="2020-01-01", end_date=None):
    df = fdr.DataReader(ticker, start_date, end_date)
    df = df.reset_index().rename(columns={"index": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    return df

def _keep_weekdays(df, date_col="Date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df[df[date_col].dt.weekday < 5].sort_values(date_col).reset_index(drop=True)
    return df

# ── 2. ETF 피처 생성 (기술지표 포함) ─────────────────────────────────────────
def make_target_etf_features(etf_df, prefix, cfg=None):
    df    = etf_df.copy().sort_values("Date").reset_index(drop=True)
    close = df["Adj Close"]
    high  = df["High"]  if "High"  in df.columns else close
    low   = df["Low"]   if "Low"   in df.columns else close
    volume= df["Volume"]

    r = pd.DataFrame()
    r["Date"]                       = df["Date"]
    r[f"{prefix}_adj_close"]        = close

    # 수익률
    r[f"{prefix}_ret_1d"]           = close.pct_change(1)
    r[f"{prefix}_ret_5d"]           = close.pct_change(5)
    r[f"{prefix}_ret_20d"]          = close.pct_change(20)

    # 이동평균 대비
    ma5  = close.rolling(5).mean()
    ma20 = close.rolling(20).mean()
    ma60 = close.rolling(60).mean()
    r[f"{prefix}_ma5_ratio"]        = close / ma5  - 1
    r[f"{prefix}_ma20_ratio"]       = close / ma20 - 1
    r[f"{prefix}_ma60_ratio"]       = close / ma60 - 1

    # 변동성
    r[f"{prefix}_vol_20d"]          = r[f"{prefix}_ret_1d"].rolling(20).std()

    # 거래량
    vol_ma20 = volume.rolling(20).mean()
    r[f"{prefix}_volume_ratio_20d"] = volume / vol_ma20 - 1

    # ── 추가 기술지표 ──
    if cfg is None or cfg.get("use_rsi", True):
        period = (cfg or {}).get("rsi_period", 14)
        r[f"{prefix}_rsi_{period}"]  = _rsi(close, period)

    if cfg is None or cfg.get("use_macd", True):
        fast = (cfg or {}).get("macd_fast", 12)
        slow = (cfg or {}).get("macd_slow", 26)
        sig  = (cfg or {}).get("macd_signal", 9)
        ema_f = close.ewm(span=fast, adjust=False).mean()
        ema_s = close.ewm(span=slow, adjust=False).mean()
        macd  = ema_f - ema_s
        macd_sig = macd.ewm(span=sig, adjust=False).mean()
        r[f"{prefix}_macd"]          = macd
        r[f"{prefix}_macd_hist"]     = macd - macd_sig
        r[f"{prefix}_macd_signal"]   = macd_sig

    if cfg is None or cfg.get("use_bollinger", True):
        p   = (cfg or {}).get("bollinger_period", 20)
        std = (cfg or {}).get("bollinger_std", 2.0)
        ma  = close.rolling(p).mean()
        sd  = close.rolling(p).std()
        upper = ma + std * sd
        lower = ma - std * sd
        band  = (upper - lower).replace(0, np.nan)
        r[f"{prefix}_bb_pct"]        = (close - lower) / band
        r[f"{prefix}_bb_width"]      = band / ma

    if cfg is None or cfg.get("use_atr", True):
        p = (cfg or {}).get("atr_period", 14)
        atr_val = _atr(high, low, close, p)
        r[f"{prefix}_atr_{p}"]       = atr_val
        r[f"{prefix}_atr_{p}_ratio"] = atr_val / close

    if cfg is None or cfg.get("use_52w", True):
        r[f"{prefix}_52w_high_ratio"] = close / close.rolling(252).max() - 1
        r[f"{prefix}_52w_low_ratio"]  = close / close.rolling(252).min() - 1

    return r

# ── 3. 외부지표 피처 ──────────────────────────────────────────────────────────
def make_external_features(raw_df, name, feature_type="price"):
    df    = raw_df.copy().sort_values("Date").reset_index(drop=True)
    close = df["Adj Close"]
    r     = pd.DataFrame()
    r["Date"] = df["Date"]

    if feature_type == "price":
        r[f"{name}_ret_5d"]    = close.pct_change(5)
        r[f"{name}_ret_20d"]   = close.pct_change(20)
    elif feature_type == "risk":
        r[f"{name}_level"]     = close
        r[f"{name}_chg_5d"]    = close.diff(5)
        r[f"{name}_chg_20d"]   = close.diff(20)
    elif feature_type == "rate":
        r[f"{name}_level"]     = close
        r[f"{name}_diff_5d"]   = close.diff(5)
        r[f"{name}_diff_20d"]  = close.diff(20)
    else:
        raise ValueError(f"Unknown feature_type: {feature_type}")
    return r

# ── 4. Base Feature Dataset ───────────────────────────────────────────────────
def make_base_feature_dataset(etf_code, external_tickers, external_feature_types,
                               start_date="2020-01-01", end_date=None, feat_cfg=None):
    etf_raw = _keep_weekdays(load_price_data(etf_code, start_date, end_date))
    base_df = make_target_etf_features(etf_raw, prefix=etf_code, cfg=feat_cfg)
    base_df = _keep_weekdays(base_df)

    for name, ticker in external_tickers.items():
        print(f"  Loading {name} ({ticker})")
        try:
            raw    = _keep_weekdays(load_price_data(ticker, start_date, end_date))
            ftype  = external_feature_types.get(name, "price")
            ext    = _keep_weekdays(make_external_features(raw, name, ftype))
            base_df= base_df.merge(ext, on="Date", how="left")
        except Exception as e:
            print(f"  [SKIP] {name}: {e}")

    base_df = _keep_weekdays(base_df)

    etf_prefix   = f"{etf_code}_"
    external_cols= [c for c in base_df.columns if c != "Date" and not c.startswith(etf_prefix)]
    base_df[external_cols] = base_df[external_cols].ffill()

    close_col   = f"{etf_code}_adj_close"
    feature_cols= [c for c in base_df.columns if c not in ["Date", close_col]]
    return base_df, feature_cols, close_col

# ── 5. VIF 제거 ───────────────────────────────────────────────────────────────
def reduce_features_by_vif(df, feature_cols, vif_threshold=30.0, verbose=True):
    numeric_cols = [c for c in feature_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    work = df[numeric_cols].replace([np.inf, -np.inf], np.nan).dropna(axis=0).copy()

    nunique = work.nunique()
    remaining = [c for c in numeric_cols if nunique.get(c, 0) > 1]
    removed = []

    while True:
        if len(remaining) <= 1: break
        X  = work[remaining].copy()
        sc = StandardScaler()
        Xs = sc.fit_transform(X)
        vifs = []
        for i, col in enumerate(remaining):
            try:    v = variance_inflation_factor(Xs, i)
            except: v = np.inf
            vifs.append((col, v))
        worst_col, worst_v = max(vifs, key=lambda x: x[1])
        if verbose: print(f"  VIF max: {worst_v:.1f}  ({worst_col})")
        if worst_v <= vif_threshold: break
        remaining.remove(worst_col)
        removed.append({"removed_feature": worst_col, "vif": worst_v})

    print(f"  VIF 제거: {len(removed)}개 / 남은 피처: {len(remaining)}개")
    return remaining, pd.DataFrame(removed)

# ── 6. Target 생성 ────────────────────────────────────────────────────────────
def add_target_column(df, close_col, n_days=5, threshold=0.05, target_col=None):
    r = df.copy().sort_values("Date").reset_index(drop=True)
    if target_col is None:
        target_col = f"target_{n_days}d_up_{int(threshold*100)}pct"
    future_close = r[close_col].shift(-n_days)
    r[f"future_ret_{n_days}d"] = future_close / r[close_col] - 1
    r[target_col] = np.where(r[f"future_ret_{n_days}d"] >= threshold, 1, 0)
    r.loc[r[f"future_ret_{n_days}d"].isna(), target_col] = np.nan
    return r, target_col

# ── 7. Lag 탐색 & 적용 ───────────────────────────────────────────────────────
def find_best_lag_by_feature(df, feature_cols, target_col, lag_days, date_col="Date"):
    records = []
    for col in feature_cols:
        if col not in df.columns: continue
        for lag in lag_days:
            tmp = df[[date_col, col, target_col]].copy()
            tmp[f"{col}_lag{lag}"] = tmp[col].shift(lag)
            tmp = tmp[[f"{col}_lag{lag}", target_col]].replace([np.inf,-np.inf], np.nan).dropna()
            if len(tmp) < 30: continue
            x = tmp[f"{col}_lag{lag}"]
            corr = x.corr(tmp[target_col]) if x.nunique() > 1 else np.nan
            records.append({"feature": col, "lag": lag, "corr": corr,
                            "abs_corr": abs(corr) if pd.notna(corr) else np.nan})
    lag_df = pd.DataFrame(records)
    if lag_df.empty: raise ValueError("lag 탐색 결과 없음")
    best_lag_df = (lag_df.sort_values(["feature","abs_corr"], ascending=[True,False])
                         .groupby("feature", as_index=False).head(1)
                         .sort_values("abs_corr", ascending=False).reset_index(drop=True))
    return lag_df, best_lag_df

def make_lagged_dataset(df, best_lag_df, target_col=None, close_col=None,
                        n_days=5, date_col="Date", drop_target_na=True):
    r = pd.DataFrame()
    r[date_col] = df[date_col]
    if close_col and close_col in df.columns:
        r[close_col] = df[close_col]
    fut = f"future_ret_{n_days}d"
    if fut in df.columns: r[fut] = df[fut]
    if target_col and target_col in df.columns: r[target_col] = df[target_col]

    lagged_cols = []
    for _, row in best_lag_df.iterrows():
        feat = row["feature"]; lag = int(row["lag"])
        if feat not in df.columns: continue
        lc = f"{feat}_lag{lag}"
        r[lc] = df[feat].shift(lag)
        lagged_cols.append(lc)

    r = r.replace([np.inf, -np.inf], np.nan)
    if drop_target_na and target_col and target_col in r.columns:
        r = r.dropna(subset=[target_col] + lagged_cols).reset_index(drop=True)
    else:
        r = r.reset_index(drop=True)
    return r, lagged_cols

# ── 8. Permutation Importance (feature selection용 RF) ────────────────────────
def run_permutation_importance(lagged_df, feature_cols, target_col,
                                n_rf_runs=3, n_repeats=5, random_state=42):
    df = lagged_df.copy()
    X  = df[feature_cols].replace([np.inf,-np.inf], np.nan)
    tmp= pd.concat([X, df[target_col]], axis=1).dropna().copy()
    X  = tmp[feature_cols].copy()
    y  = tmp[target_col].astype(int)

    all_imp = []
    for run in range(n_rf_runs):
        rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    random_state=random_state+run, n_jobs=-1)
        rf.fit(X, y)
        pi = permutation_importance(rf, X, y, n_repeats=n_repeats,
                                    random_state=random_state+run, n_jobs=-1)
        for fi, col in enumerate(feature_cols):
            for val in pi.importances[fi]:
                all_imp.append({"feature": col, "importance": val})

    imp_df = pd.DataFrame(all_imp)
    agg = (imp_df.groupby("feature")["importance"]
                 .agg(mean_importance="mean", std_importance="std")
                 .reset_index()
                 .sort_values("mean_importance", ascending=False)
                 .reset_index(drop=True))
    agg["score"] = agg["mean_importance"] - agg["std_importance"].fillna(0)
    return agg

# ── 9. 지표 계산 ──────────────────────────────────────────────────────────────
def safe_binary_metrics(y_true, pred, pred_proba=None):
    y_true = pd.Series(y_true).astype(int)
    pred   = pd.Series(pred).astype(int)
    out = {
        "eval_count":    int(len(y_true)),
        "actual_1_count":int((y_true==1).sum()),
        "pred_1_count":  int((pred==1).sum()),
        "accuracy":      accuracy_score(y_true, pred) if len(y_true) else np.nan,
        "precision":     precision_score(y_true, pred, zero_division=0),
        "recall":        recall_score(y_true, pred, zero_division=0),
        "f1":            f1_score(y_true, pred, zero_division=0),
    }
    if pred_proba is not None and len(y_true) > 0 and y_true.nunique() == 2:
        try: out["roc_auc"] = roc_auc_score(y_true, pred_proba)
        except: out["roc_auc"] = np.nan
    return out

# ── 10. 모델 생성 (XGBoost/LightGBM 포함) ───────────────────────────────────
def make_classifier(model_name, random_state=42, model_params=None,
                    pos_count=None, neg_count=None):
    p = model_params or {}

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators   = p.get("n_estimators", 500),
            max_depth      = p.get("max_depth", None),
            min_samples_split = p.get("min_samples_split", 2),
            min_samples_leaf  = p.get("min_samples_leaf", 1),
            max_features   = p.get("max_features", "sqrt"),
            class_weight   = p.get("class_weight", "balanced"),
            random_state   = random_state, n_jobs=-1)

    if model_name == "extra_trees":
        return ExtraTreesClassifier(
            n_estimators   = p.get("n_estimators", 500),
            max_depth      = p.get("max_depth", None),
            min_samples_split = p.get("min_samples_split", 2),
            min_samples_leaf  = p.get("min_samples_leaf", 1),
            max_features   = p.get("max_features", "sqrt"),
            class_weight   = p.get("class_weight", "balanced"),
            random_state   = random_state, n_jobs=-1)

    if model_name == "gradient_boosting":
        return GradientBoostingClassifier(
            n_estimators   = p.get("n_estimators", 300),
            learning_rate  = p.get("learning_rate", 0.05),
            max_depth      = p.get("max_depth", 3),
            min_samples_leaf = p.get("min_samples_leaf", 1),
            random_state   = random_state)

    if model_name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            max_iter       = p.get("max_iter", 300),
            learning_rate  = p.get("learning_rate", 0.05),
            max_leaf_nodes = p.get("max_leaf_nodes", 31),
            min_samples_leaf = p.get("min_samples_leaf", 20),
            class_weight   = "balanced",
            random_state   = random_state)

    if model_name == "xgboost":
        scale_pw = (neg_count / pos_count) if (pos_count and neg_count and pos_count > 0) else 1.0
        return xgb.XGBClassifier(
            n_estimators     = p.get("n_estimators", 300),
            learning_rate    = p.get("learning_rate", 0.05),
            max_depth        = p.get("max_depth", 4),
            subsample        = p.get("subsample", 0.8),
            colsample_bytree = p.get("colsample_bytree", 0.8),
            scale_pos_weight = scale_pw,
            random_state     = random_state,
            eval_metric      = "logloss",
            verbosity        = 0, n_jobs=-1)

    if model_name == "lightgbm":
        return lgb.LGBMClassifier(
            n_estimators     = p.get("n_estimators", 300),
            learning_rate    = p.get("learning_rate", 0.05),
            max_depth        = p.get("max_depth", -1),
            num_leaves       = p.get("num_leaves", 31),
            subsample        = p.get("subsample", 0.8),
            colsample_bytree = p.get("colsample_bytree", 0.8),
            is_unbalance     = True,
            random_state     = random_state,
            verbose          = -1, n_jobs=-1)

    raise ValueError(f"Unknown model: {model_name}")

def suggest_model_params(trial, model_name):
    if model_name in ["random_forest", "extra_trees"]:
        return {
            "n_estimators":    trial.suggest_int(f"{model_name}_n_est", 200, 800, step=100),
            "max_depth":       trial.suggest_categorical(f"{model_name}_max_depth", [None,3,5,7,10]),
            "min_samples_split": trial.suggest_int(f"{model_name}_mss", 2, 10),
            "min_samples_leaf":  trial.suggest_int(f"{model_name}_msl", 1, 10),
            "max_features":    trial.suggest_categorical(f"{model_name}_mf", ["sqrt","log2"]),
            "class_weight":    trial.suggest_categorical(f"{model_name}_cw",
                                   ["balanced","balanced_subsample"]),
        }
    if model_name == "gradient_boosting":
        return {
            "n_estimators":    trial.suggest_int("gb_n_est", 100, 500, step=100),
            "learning_rate":   trial.suggest_float("gb_lr", 0.01, 0.2, log=True),
            "max_depth":       trial.suggest_int("gb_depth", 2, 5),
            "min_samples_leaf":trial.suggest_int("gb_msl", 1, 20),
        }
    if model_name == "hist_gradient_boosting":
        return {
            "max_iter":        trial.suggest_int("hgb_iter", 100, 500, step=100),
            "learning_rate":   trial.suggest_float("hgb_lr", 0.01, 0.2, log=True),
            "max_leaf_nodes":  trial.suggest_int("hgb_leaves", 15, 63),
            "min_samples_leaf":trial.suggest_int("hgb_msl", 10, 50),
        }
    if model_name == "xgboost":
        return {
            "n_estimators":    trial.suggest_int("xgb_n_est", 100, 600, step=100),
            "learning_rate":   trial.suggest_float("xgb_lr", 0.01, 0.2, log=True),
            "max_depth":       trial.suggest_int("xgb_depth", 3, 8),
            "subsample":       trial.suggest_float("xgb_sub", 0.6, 1.0, step=0.1),
            "colsample_bytree":trial.suggest_float("xgb_col", 0.6, 1.0, step=0.1),
        }
    if model_name == "lightgbm":
        return {
            "n_estimators":    trial.suggest_int("lgb_n_est", 100, 600, step=100),
            "learning_rate":   trial.suggest_float("lgb_lr", 0.01, 0.2, log=True),
            "num_leaves":      trial.suggest_int("lgb_leaves", 15, 127),
            "subsample":       trial.suggest_float("lgb_sub", 0.6, 1.0, step=0.1),
            "colsample_bytree":trial.suggest_float("lgb_col", 0.6, 1.0, step=0.1),
        }
    raise ValueError(model_name)

def evaluate_model_on_period(train_df, eval_df, feature_cols, target_col, close_col,
                              n_days, model_name, model_params, pred_threshold,
                              random_state=42):
    keep_train = [target_col] + feature_cols
    tr = train_df[keep_train].replace([np.inf,-np.inf], np.nan).dropna().copy()
    keep_eval  = ["Date", target_col, f"future_ret_{n_days}d", close_col] + feature_cols
    ev = eval_df[[c for c in keep_eval if c in eval_df.columns]].replace([np.inf,-np.inf], np.nan).dropna().copy()

    if len(tr) == 0 or len(ev) == 0:
        raise ValueError("train/eval 비어있음")
    if tr[target_col].nunique() < 2:
        raise ValueError("train target 단일 class")

    X_tr = tr[feature_cols]; y_tr = tr[target_col].astype(int)
    X_ev = ev[feature_cols]; y_ev = ev[target_col].astype(int)

    pos = int((y_tr == 1).sum()); neg = int((y_tr == 0).sum())
    model = make_classifier(model_name, random_state=random_state,
                            model_params=model_params, pos_count=pos, neg_count=neg)

    # GradientBoosting: class_weight 미지원 → sample_weight 사용
    if model_name == "gradient_boosting":
        sw = compute_sample_weight("balanced", y_tr)
        model.fit(X_tr, y_tr, sample_weight=sw)
    else:
        model.fit(X_tr, y_tr)

    pred_proba = model.predict_proba(X_ev)[:,1] if hasattr(model,"predict_proba") else model.predict(X_ev).astype(float)
    pred       = (pred_proba >= pred_threshold).astype(int)
    metrics    = safe_binary_metrics(y_ev, pred, pred_proba)

    pred_df = ev[["Date", close_col, f"future_ret_{n_days}d", target_col]].copy()
    pred_df["pred_proba"] = pred_proba
    pred_df["pred"]       = pred
    return model, metrics, pred_df

def compute_objective_score(metrics, pred_df, objective_metric, n_days,
                             min_eval=30, min_pred1=3):
    ec   = int(metrics.get("eval_count",0) or 0)
    a1   = int(metrics.get("actual_1_count",0) or 0)
    p1   = int(metrics.get("pred_1_count",0) or 0)
    if ec < min_eval or p1 < min_pred1: return 0.0

    pos_rate  = a1 / ec
    precision = float(metrics.get("precision",0) or 0)
    recall    = float(metrics.get("recall",0) or 0)
    f1        = float(metrics.get("f1",0) or 0)
    pl        = max(0.0, precision - pos_rate)
    fl        = max(0.0, f1 - pos_rate)

    if objective_metric == "precision_lift_recall": return pl * recall
    if objective_metric == "precision_lift":        return pl
    if objective_metric == "f1_lift":               return fl
    if objective_metric == "return_score":
        strat = np.where(pred_df["pred"]==1, pred_df[f"future_ret_{n_days}d"], 0.0)
        cret  = float(np.prod(1 + strat) - 1)
        return cret + 0.5 * pl * recall
    raise ValueError(objective_metric)

def _metrics_extra(metrics, n_days, pred_df=None):
    ec = int(metrics.get("eval_count",0) or 0)
    a1 = int(metrics.get("actual_1_count",0) or 0)
    pos_rate  = a1 / ec if ec > 0 else np.nan
    precision = float(metrics.get("precision",0) or 0)
    recall    = float(metrics.get("recall",0) or 0)
    f1        = float(metrics.get("f1",0) or 0)
    p1        = int(metrics.get("pred_1_count",0) or 0)
    return {
        "positive_rate":        pos_rate,
        "precision_lift":       precision - pos_rate if pd.notna(pos_rate) else np.nan,
        "f1_lift":              f1 - pos_rate        if pd.notna(pos_rate) else np.nan,
        "precision_lift_recall":max(0, precision-pos_rate)*recall if pd.notna(pos_rate) else np.nan,
        "pred_1_ratio":         p1/ec if ec > 0 else np.nan,
    }

print("공통 함수 로드 완료")


공통 함수 로드 완료


In [3]:
def make_oos_windows(base_date, start_date, optuna_valid_months, sim_test_months):
    base  = pd.to_datetime(base_date)
    start = pd.to_datetime(start_date)
    base_p= base.to_period("M")

    sim_start_p   = base_p  - (sim_test_months - 1)
    valid_end_p   = sim_start_p - 1
    valid_start_p = valid_end_p - (optuna_valid_months - 1)
    train_end_p   = valid_start_p - 1

    return {
        "train_start":        start,
        "train_end":          train_end_p.to_timestamp(how="end").normalize(),
        "optuna_valid_start": valid_start_p.to_timestamp(how="start"),
        "optuna_valid_end":   valid_end_p.to_timestamp(how="end").normalize(),
        "sim_test_start":     sim_start_p.to_timestamp(how="start"),
        "sim_test_end":       base,
        "base_date":          base,
    }

WINDOWS = make_oos_windows(BASE_DATE, START_DATE, OPTUNA_VALID_MONTHS, SIM_TEST_MONTHS)
for k, v in WINDOWS.items():
    print(f"  {k:22s}: {str(v)[:10]}")


  train_start           : 2017-01-01
  train_end             : 2024-12-31
  optuna_valid_start    : 2025-01-01
  optuna_valid_end      : 2025-06-30
  sim_test_start        : 2025-07-01
  sim_test_end          : 2026-05-31
  base_date             : 2026-05-31


In [4]:
feat_cfg = {
    "use_rsi": USE_RSI, "rsi_period": RSI_PERIOD,
    "use_macd": USE_MACD, "macd_fast": MACD_FAST, "macd_slow": MACD_SLOW, "macd_signal": MACD_SIG,
    "use_bollinger": USE_BB, "bollinger_period": BB_PERIOD, "bollinger_std": BB_STD,
    "use_atr": USE_ATR, "atr_period": ATR_PERIOD,
    "use_52w": USE_52W,
}

if LOAD_FROM_CACHE:
    base_df, raw_feature_cols, close_col = load_cache("base_dataset")
else:
    print("Base feature dataset 생성 중...")
    base_df, raw_feature_cols, close_col = make_base_feature_dataset(
        etf_code=ETF_CODE,
        external_tickers=EXTERNAL_TICKERS,
        external_feature_types=EXTERNAL_FEATURE_TYPES,
        start_date=START_DATE,
        end_date=END_DATE,
        feat_cfg=feat_cfg,
    )
    save_cache("base_dataset", (base_df, raw_feature_cols, close_col))

print(f"\nbase_df shape: {base_df.shape}")
print(f"피처 수: {len(raw_feature_cols)}")
print(f"기간: {base_df['Date'].min().date()} ~ {base_df['Date'].max().date()}")


[CACHE LOADED] cache_SMH_20260531/base_dataset.pkl

base_df shape: (2374, 40)
피처 수: 38
기간: 2017-01-03 ~ 2026-06-12


In [5]:
if LOAD_FROM_CACHE:
    vif_feature_cols = load_cache("vif_feature_cols")
else:
    train_base_df = base_df[
        (base_df["Date"] >= WINDOWS["train_start"]) &
        (base_df["Date"] <= WINDOWS["train_end"])
    ].copy()

    print(f"VIF 계산 대상: {train_base_df.shape}")
    vif_feature_cols, removed_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=raw_feature_cols,
        vif_threshold=VIF_THRESHOLD,
    )
    save_cache("vif_feature_cols", vif_feature_cols)

print(f"VIF 통과 피처: {len(vif_feature_cols)}개")


[CACHE LOADED] cache_SMH_20260531/vif_feature_cols.pkl
VIF 통과 피처: 34개


In [6]:
feature_cache = {}
if LOAD_FROM_CACHE:
    try:
        feature_cache = load_cache("feature_cache")
        print(f"feature_cache 로드: {len(feature_cache)}가지 조합")
    except Exception as e:
        print(f"feature_cache 없음, 새로 생성")


def _gc(msg):
    gc.collect()
    print(f"  [GC] {msg}")

def build_feature_artifacts(n_days, target_return_threshold, include_infer_df=False):
    key = (int(n_days), float(target_return_threshold))
    if key in feature_cache:
        art = feature_cache[key]
        if include_infer_df and "all_lagged_infer_df" not in art:
            td, _ = add_target_column(base_df, close_col, n_days, target_return_threshold)
            art["all_lagged_infer_df"], _ = make_lagged_dataset(
                td, art["best_lag_df"], target_col=art["target_col"],
                close_col=close_col, n_days=n_days, drop_target_na=False)
            del td; _gc("infer target_df deleted")
        return art

    print(f"  [build] n_days={n_days}, threshold={target_return_threshold}")

    target_df, target_col = add_target_column(base_df, close_col, n_days, target_return_threshold)

    fs_end   = WINDOWS["train_end"]
    fs_start = fs_end - pd.DateOffset(years=FEATURE_SELECT_YEARS)
    fs_df    = target_df[(target_df["Date"]>=fs_start)&(target_df["Date"]<=fs_end)].copy()
    print(f"    feature select 기간: {fs_df['Date'].min().date()} ~ {fs_df['Date'].max().date()}")

    _, best_lag_df = find_best_lag_by_feature(
        fs_df, vif_feature_cols, target_col, LAG_DAYS)
    del fs_df; _gc("fs_df deleted")

    all_lagged_eval_df, lagged_cols = make_lagged_dataset(
        target_df, best_lag_df, target_col=target_col,
        close_col=close_col, n_days=n_days, drop_target_na=True)

    fs_lag_df = all_lagged_eval_df[
        (all_lagged_eval_df["Date"]>=fs_start)&(all_lagged_eval_df["Date"]<=fs_end)].copy()

    if fs_lag_df[target_col].nunique() < 2:
        raise ValueError("Feature selection target 단일 class")

    imp_df = run_permutation_importance(
        fs_lag_df, lagged_cols, target_col, N_RF_RUNS, N_REPEATS, RANDOM_STATE)

    top_cols_max = imp_df["feature"].head(TOP_N_MAX).tolist()

    art = {
        "n_days": int(n_days), "target_return_threshold": target_return_threshold,
        "target_col": target_col, "best_lag_df": best_lag_df,
        "all_lagged_eval_df": all_lagged_eval_df, "lagged_feature_cols": lagged_cols,
        "importance_df": imp_df, "top_feature_cols_max": top_cols_max,
    }
    if include_infer_df:
        art["all_lagged_infer_df"], _ = make_lagged_dataset(
            target_df, best_lag_df, target_col=target_col,
            close_col=close_col, n_days=n_days, drop_target_na=False)

    feature_cache[key] = art
    del target_df, fs_lag_df; _gc(f"deleted after build n_days={n_days} th={target_return_threshold}")
    return art

print("Feature cache 함수 준비 완료")


[CACHE LOADED] cache_SMH_20260531/feature_cache.pkl
feature_cache 로드: 18가지 조합
Feature cache 함수 준비 완료


In [7]:
def run_stage1_grid_search(objective_metric, n_seeds=3, random_state=42):
    """
    Stage 1: n_days x target_return_threshold 전수조사 (18가지 조합).

    Optuna 대신 grid search를 쓰는 이유:
    - 탐색 공간이 n_days(3) x threshold(6) = 18가지로 고정
    - Optuna TPE는 좋아 보이는 조합을 반복 재시도해서 결과가 중복됨
    - 전수조사하면 모든 조합의 성능을 정확히 비교 가능

    n_seeds: 동일 조합을 다른 random_state로 n번 반복해 점수를 안정화
    """
    from itertools import product
    rows = []

    combos = list(product(N_DAYS_CANDIDATES, TARGET_RETURN_THRESHOLD_CANDIDATES))
    print(f"총 {len(combos)}가지 조합 x {n_seeds} seeds = {len(combos)*n_seeds}회 평가")

    for i, (n_days, threshold) in enumerate(combos):
        print(f"  [{i+1}/{len(combos)}] n_days={n_days}, threshold={threshold}", end=" ")
        seed_scores = []
        try:
            art = build_feature_artifacts(n_days, threshold)
            tc  = art["target_col"]
            ev  = art["all_lagged_eval_df"]
            fc  = art["top_feature_cols_max"][:S1_TOP_N]

            if not fc:
                print("→ 피처 없음 skip")
                continue

            tr_df = ev[(ev["Date"]>=WINDOWS["train_start"])&(ev["Date"]<=WINDOWS["train_end"])]
            va_df = ev[(ev["Date"]>=WINDOWS["optuna_valid_start"])&(ev["Date"]<=WINDOWS["optuna_valid_end"])]
            params = {"n_estimators": S1_N_EST, "class_weight": "balanced"}

            for s in range(n_seeds):
                _, metrics, pred_df = evaluate_model_on_period(
                    tr_df, va_df, fc, tc, close_col, n_days,
                    S1_MODEL, params, pred_threshold=0.5,
                    random_state=random_state + s * 100)
                score = compute_objective_score(metrics, pred_df, objective_metric,
                                                n_days, S1_MIN_EVAL, S1_MIN_PRED1)
                seed_scores.append(score)

            mean_score = float(pd.Series(seed_scores).mean())
            std_score  = float(pd.Series(seed_scores).std())
            print(f"→ score={mean_score:.4f} (±{std_score:.4f})")

            # 마지막 seed 기준 metrics 저장
            rows.append({
                "n_days": n_days, "threshold": threshold,
                "score": mean_score, "score_std": std_score,
                **metrics, **_metrics_extra(metrics, n_days, pred_df),
            })

        except Exception as e:
            print(f"→ ERROR: {e}")
            rows.append({"n_days": n_days, "threshold": threshold,
                         "score": 0.0, "error": str(e)})

    trials_df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    best = trials_df.iloc[0]
    return {
        "trials_df": trials_df,
        "best_n_days":   int(best["n_days"]),
        "best_threshold": float(best["threshold"]),
        "best_score":    float(best["score"]),
    }

_s1_pkl = CACHE_DIR / "stage1_result.pkl"
if LOAD_FROM_CACHE and _s1_pkl.exists():
    stage1_result = load_cache("stage1_result")
else:
    print("Stage 1 Grid Search 시작...")
    stage1_result = run_stage1_grid_search(
        objective_metric=S1_OBJ_METRIC,
        n_seeds=S1_N_SEEDS,
        random_state=RANDOM_STATE,
    )
    save_cache("stage1_result", stage1_result)

print(f"\nStage 1 Best → n_days={stage1_result['best_n_days']}, "
      f"threshold={stage1_result['best_threshold']}, score={stage1_result['best_score']:.4f}")

stage1_result["trials_df"].to_csv(OUTPUT_DIR / "stage1_grid_results.csv", index=False)
display_df(stage1_result["trials_df"])


[CACHE LOADED] cache_SMH_20260531/stage1_result.pkl

Stage 1 Best → n_days=10, threshold=0.04, score=0.0011


,n_days,threshold,score,score_std,eval_count,actual_1_count,pred_1_count,accuracy,precision,recall,f1,roc_auc,positive_rate,precision_lift,f1_lift,precision_lift_recall,pred_1_ratio
0,10,0.04,0.001063,0.001840,122,54,3,0.549180,0.333333,0.018519,0.035088,0.408088,0.442623,-0.109290,-0.407535,0.000000,0.024590
1,20,0.05,0.000488,0.000845,122,56,3,0.532787,0.333333,0.017857,0.033898,0.400027,0.459016,-0.125683,-0.425118,0.000000,0.024590
2,5,0.04,0.000000,0.000000,122,31,2,0.762295,1.000000,0.064516,0.121212,0.489011,0.254098,0.745902,-0.132886,0.048123,0.016393
3,5,0.05,0.000000,0.000000,122,25,1,0.786885,0.000000,0.000000,0.000000,0.484330,0.204918,-0.204918,-0.204918,0.000000,0.008197
4,5,0.06,0.000000,0.000000,122,19,0,0.844262,0.000000,0.000000,0.000000,0.539601,0.155738,-0.155738,-0.155738,0.000000,0.000000
5,5,0.07,0.000000,0.000000,122,14,0,0.885246,0.000000,0.000000,0.000000,0.577712,0.114754,-0.114754,-0.114754,0.000000,0.000000
6,10,0.05,0.000000,0.000000,122,43,0,0.647541,0.000000,0.000000,0.000000,0.408154,0.352459,-0.352459,-0.352459,0.000000,0.000000
7,10,0.06,0.000000,0.000000,122,33,1,0.721311,0.000000,0.000000,0.000000,0.362104,0.270492,-0.270492,-0.270492,0.000000,0.008197
8,10,0.07,0.000000,0.000000,122,29,2,0.745902,0.000000,0.000000,0.000000,0.417686,0.237705,-0.237705,-0.237705,0.000000,0.016393
9,20,0.04,0.000000,0.000000,122,58,82,0.393443,0.402439,0.568966,0.471429,0.418238,0.475410,-0.072971,-0.003981,0.000000,0.672131


In [8]:
def run_stage2_optuna(n_days, threshold, n_trials, objective_metric, random_state=42):
    '''Stage 2: 고정된 n_days+threshold에서 모델·하이퍼파라미터 탐색'''
    art = build_feature_artifacts(n_days, threshold)
    tc  = art["target_col"]
    ev  = art["all_lagged_eval_df"]
    all_fc = art["top_feature_cols_max"]

    rows = []

    def objective(trial):
        top_n = trial.suggest_int("top_n", S2_TOP_N_RANGE[0], min(S2_TOP_N_RANGE[1], len(all_fc)))
        fc    = all_fc[:top_n]
        mname = trial.suggest_categorical("model_name", S2_MODELS)
        mparams = suggest_model_params(trial, mname)
        pth   = trial.suggest_float("pred_threshold", S2_THRESH_RANGE[0],
                                    S2_THRESH_RANGE[1], step=S2_THRESH_STEP)

        tr_df = ev[(ev["Date"]>=WINDOWS["train_start"])&(ev["Date"]<=WINDOWS["train_end"])]
        va_df = ev[(ev["Date"]>=WINDOWS["optuna_valid_start"])&(ev["Date"]<=WINDOWS["optuna_valid_end"])]

        try:
            _, metrics, pred_df = evaluate_model_on_period(
                tr_df, va_df, fc, tc, close_col, n_days,
                mname, mparams, pth, random_state=random_state)
            score = compute_objective_score(metrics, pred_df, objective_metric,
                                            n_days, S2_MIN_EVAL, S2_MIN_PRED1)
            row = {"trial": trial.number, "score": score, "model": mname,
                   "top_n": top_n, "pred_threshold": pth,
                   **metrics, **_metrics_extra(metrics, n_days, pred_df),
                   "model_params": json.dumps(mparams, default=str)}
            rows.append(row)
            trial.set_user_attr("model_params", mparams)
            trial.set_user_attr("metrics", metrics)
            return score
        except Exception as e:
            rows.append({"trial": trial.number, "score": 0.0, "error": str(e),
                         "model": mname})
            return 0.0

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    trials_df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    bt = study.best_trial

    best_config = {
        "n_days":           n_days,
        "target_return_threshold": threshold,
        "target_col":       art["target_col"],
        "model_name":       bt.params["model_name"],
        "model_params":     bt.user_attrs["model_params"],
        "top_n":            bt.params["top_n"],
        "pred_threshold":   bt.params["pred_threshold"],
        "score":            bt.value,
        "objective_metric": objective_metric,
    }
    return {"study": study, "trials_df": trials_df, "best_config": best_config,
            "artifacts": art}

_s2_pkl = CACHE_DIR / "stage2_result.pkl"
if LOAD_FROM_CACHE and _s2_pkl.exists():
    stage2_result = load_cache("stage2_result")
else:
    print(f"Stage 2 Optuna 시작 ({S2_N_TRIALS} trials) ...")
    stage2_result = run_stage2_optuna(
        stage1_result["best_n_days"],
        stage1_result["best_threshold"],
        S2_N_TRIALS, S2_OBJ_METRIC, RANDOM_STATE,
    )
    save_cache("stage2_result", stage2_result)
    save_cache("feature_cache", feature_cache)

best_cfg = stage2_result["best_config"]
print("\nBest config:")
print(json.dumps(best_cfg, ensure_ascii=False, indent=2, default=str))

stage2_result["trials_df"].to_csv(OUTPUT_DIR / "stage2_optuna_trials.csv", index=False)
display_df(stage2_result["trials_df"], 15)


Stage 2 Optuna 시작 (200 trials) ...


[CACHE SAVED] cache_SMH_20260531/stage2_result.pkl
[CACHE SAVED] cache_SMH_20260531/feature_cache.pkl

Best config:
{
  "n_days": 10,
  "target_return_threshold": 0.04,
  "target_col": "target_10d_up_4pct",
  "model_name": "lightgbm",
  "model_params": {
    "n_estimators": 400,
    "learning_rate": 0.04532714439457619,
    "num_leaves": 68,
    "subsample": 0.7,
    "colsample_bytree": 0.8
  },
  "top_n": 5,
  "pred_threshold": 0.4,
  "score": 0.09264201296225696,
  "objective_metric": "precision_lift_recall"
}


,trial,score,model,top_n,pred_threshold,eval_count,actual_1_count,pred_1_count,accuracy,precision,recall,f1,roc_auc,positive_rate,precision_lift,f1_lift,precision_lift_recall,pred_1_ratio,model_params
0,155,0.092642,lightgbm,5,0.40,122,54,43,0.647541,0.627907,0.500000,0.556701,0.650871,0.442623,0.185284,0.114078,0.092642,0.352459,"{""n_estimators"": 400, ""learning_rate"": 0.04532..."
1,171,0.091853,lightgbm,6,0.40,122,54,39,0.647541,0.641026,0.462963,0.537634,0.628540,0.442623,0.198403,0.095011,0.091853,0.319672,"{""n_estimators"": 400, ""learning_rate"": 0.04993..."
2,96,0.091567,lightgbm,5,0.35,122,54,37,0.647541,0.648649,0.444444,0.527473,0.649782,0.442623,0.206026,0.084850,0.091567,0.303279,"{""n_estimators"": 300, ""learning_rate"": 0.09056..."
3,83,0.091567,lightgbm,5,0.40,122,54,37,0.647541,0.648649,0.444444,0.527473,0.657680,0.442623,0.206026,0.084850,0.091567,0.303279,"{""n_estimators"": 400, ""learning_rate"": 0.11648..."
4,163,0.091567,lightgbm,6,0.40,122,54,37,0.647541,0.648649,0.444444,0.527473,0.633170,0.442623,0.206026,0.084850,0.091567,0.303279,"{""n_estimators"": 400, ""learning_rate"": 0.05043..."
5,106,0.091493,lightgbm,8,0.35,122,54,29,0.647541,0.689655,0.370370,0.481928,0.651688,0.442623,0.247032,0.039305,0.091493,0.237705,"{""n_estimators"": 300, ""learning_rate"": 0.10207..."
6,120,0.091277,lightgbm,6,0.45,122,54,33,0.647541,0.666667,0.407407,0.505747,0.630447,0.442623,0.224044,0.063124,0.091277,0.270492,"{""n_estimators"": 400, ""learning_rate"": 0.09289..."
7,150,0.085507,lightgbm,5,0.30,122,54,44,0.639344,0.613636,0.500000,0.551020,0.659041,0.442623,0.171013,0.108397,0.085507,0.360656,"{""n_estimators"": 400, ""learning_rate"": 0.04992..."
8,141,0.084945,lightgbm,5,0.35,122,54,42,0.639344,0.619048,0.481481,0.541667,0.654956,0.442623,0.176425,0.099044,0.084945,0.344262,"{""n_estimators"": 400, ""learning_rate"": 0.05081..."
9,162,0.084434,lightgbm,5,0.40,122,54,40,0.639344,0.625000,0.462963,0.531915,0.651144,0.442623,0.182377,0.089292,0.084434,0.327869,"{""n_estimators"": 400, ""learning_rate"": 0.05158..."


In [9]:
best_artifacts = build_feature_artifacts(
    best_cfg["n_days"], best_cfg["target_return_threshold"], include_infer_df=True)

N_DAYS_BEST   = int(best_cfg["n_days"])
target_col    = best_artifacts["target_col"]
all_lagged_ev = best_artifacts["all_lagged_eval_df"]
all_lagged_in = best_artifacts["all_lagged_infer_df"]
best_fc       = best_artifacts["top_feature_cols_max"][:best_cfg["top_n"]]

print(f"n_days={N_DAYS_BEST}, threshold={best_cfg['target_return_threshold']}")
print(f"model={best_cfg['model_name']}, top_n={best_cfg['top_n']}, pred_threshold={best_cfg['pred_threshold']}")
print(f"feature 수: {len(best_fc)}")
display_df(best_artifacts["importance_df"], 20)

if not LOAD_FROM_CACHE:
    save_cache("feature_cache", feature_cache)


  [GC] infer target_df deleted
n_days=10, threshold=0.04
model=lightgbm, top_n=5, pred_threshold=0.4
feature 수: 5


,feature,mean_importance,std_importance,score
0,SMH_atr_14_ratio_lag60,0.012262,0.009269,0.002992
1,GOLD_ret_20d_lag60,0.003810,0.003380,0.000429
2,SMH_macd_hist_lag60,0.001270,0.002176,-0.000906
3,OIL_ret_20d_lag120,0.000794,0.001595,-0.000802
4,SMH_52w_low_ratio_lag40,0.000635,0.001462,-0.000827
5,SMH_ret_1d_lag60,0.000437,0.001248,-0.000811
6,SMH_volume_ratio_20d_lag5,0.000278,0.001018,-0.000740
7,VIX_chg_20d_lag1,0.000119,0.000680,-0.000561
8,SPY_ret_5d_lag60,0.000040,0.000397,-0.000357
9,SMH_atr_14_lag60,0.000040,0.000397,-0.000357


In [10]:
def summarize_sim(pred_df, target_col, n_days):
    ev = pred_df.dropna(subset=[target_col, "pred", f"future_ret_{n_days}d"]).copy()
    if len(ev) == 0:
        return pd.DataFrame([{"eval_count": 0}])
    metrics = safe_binary_metrics(ev[target_col].astype(int), ev["pred"].astype(int), ev["pred_proba"])
    ex      = _metrics_extra(metrics, n_days, ev)
    strat   = np.where(ev["pred"]==1, ev[f"future_ret_{n_days}d"], 0.0)
    strat_if= np.where(ev[target_col].astype(int)==1, ev[f"future_ret_{n_days}d"], 0.0)
    cret    = float(np.prod(1+strat)-1)
    cret_if = float(np.prod(1+strat_if)-1)
    avg_buy = float(ev.loc[ev["pred"]==1, f"future_ret_{n_days}d"].mean()) if (ev["pred"]==1).any() else np.nan
    return pd.DataFrame([{"target_col": target_col, "n_days": n_days,
                           **metrics, **ex,
                           "strategy_compound_return": cret,
                           "strategy_compound_return_if": cret_if,
                           "avg_ret_when_buy": avg_buy}])


def make_cash_simulation(pred_df, close_col, target_col, n_days,
                          take_profit_threshold,
                          initial_cash=1_000_000, buy_ratio=0.05, min_cash_ratio=0.30):
    """
    pred 기준 시뮬레이션과 actual_target_if(완벽한 예측) 기준을 동시에 추적.
    결과 컬럼:
      asset        / asset_if        : 총 자산
      cum_return   / cum_return_if   : 누적 수익률
      trade_action / trade_action_if : BUY / SELL_EXPIRE / SELL_TAKE_PROFIT / HOLD
    """
    df = pred_df.copy().sort_values("Date").reset_index(drop=True)
    min_cash = initial_cash * min_cash_ratio

    cash    = float(initial_cash)
    cash_if = float(initial_cash)
    open_lots    = []
    open_lots_if = []
    records = []

    def _process_lots(lots, price, i, tp_th):
        remaining, sell_amt, reason = [], 0.0, None
        for lot in lots:
            ret = price / lot["buy_price"] - 1
            if ret >= tp_th:
                sv = lot["qty"] * price; sell_amt += sv; reason = "TAKE_PROFIT"
            elif lot["sell_idx"] <= i:
                sv = lot["qty"] * price; sell_amt += sv
                if reason != "TAKE_PROFIT": reason = "EXPIRE"
            else:
                remaining.append(lot)
                sv = 0.0
        return remaining, sell_amt, reason

    def _buy(lots, cash_val, asset_val, price, i, buy_ratio, min_cash):
        spend = min(asset_val * buy_ratio, cash_val - min_cash)
        if spend <= 0 or price <= 0:
            return lots, cash_val, 0.0
        qty = spend / price
        cash_val -= spend
        lots.append({"buy_price": price, "qty": qty, "sell_idx": i + n_days})
        return lots, cash_val, spend

    for i, row in df.iterrows():
        date   = row["Date"]
        price  = float(row[close_col])
        actual = int(row[target_col]) if pd.notna(row.get(target_col)) else 0

        # ── pred 기준 ──
        open_lots, sell_amt, sell_rsn = _process_lots(open_lots, price, i, take_profit_threshold)
        cash += sell_amt
        holding = sum(l["qty"]*price for l in open_lots)
        asset   = cash + holding

        action = "HOLD"
        buy_amt = 0.0
        if pd.notna(row.get("pred")) and int(row["pred"]) == 1 and cash > min_cash:
            open_lots, cash, buy_amt = _buy(open_lots, cash, asset, price, i, buy_ratio, min_cash)
            action = "BUY"
        if sell_amt > 0:
            action = f"SELL_{sell_rsn}" + ("_BUY" if buy_amt > 0 else "")

        holding = sum(l["qty"]*price for l in open_lots)
        asset   = cash + holding

        # ── actual_target_if 기준 ──
        open_lots_if, sell_amt_if, sell_rsn_if = _process_lots(open_lots_if, price, i, take_profit_threshold)
        cash_if += sell_amt_if
        holding_if = sum(l["qty"]*price for l in open_lots_if)
        asset_if   = cash_if + holding_if

        action_if = "HOLD"
        buy_amt_if = 0.0
        if actual == 1 and cash_if > min_cash:
            open_lots_if, cash_if, buy_amt_if = _buy(open_lots_if, cash_if, asset_if, price, i, buy_ratio, min_cash)
            action_if = "BUY"
        if sell_amt_if > 0:
            action_if = f"SELL_{sell_rsn_if}" + ("_BUY" if buy_amt_if > 0 else "")

        holding_if = sum(l["qty"]*price for l in open_lots_if)
        asset_if   = cash_if + holding_if

        records.append({
            "Date": date, "price": price,
            "pred": row.get("pred"), "pred_proba": row.get("pred_proba"),
            "actual_target": actual,
            # pred 기준
            "trade_action": action, "cash": cash,
            "buy_amount": buy_amt, "sell_amount": sell_amt,
            "holding_value": holding, "asset": asset,
            "cum_return": asset / initial_cash - 1,
            # what-if 기준
            "trade_action_if": action_if, "cash_if": cash_if,
            "buy_amount_if": buy_amt_if, "sell_amount_if": sell_amt_if,
            "holding_value_if": holding_if, "asset_if": asset_if,
            "cum_return_if": asset_if / initial_cash - 1,
        })

    return pd.DataFrame(records)


# ── Simulation Test 실행 ──────────────────────────────────────────────────
tr_va_df = all_lagged_ev[
    (all_lagged_ev["Date"] >= WINDOWS["train_start"]) &
    (all_lagged_ev["Date"] <= WINDOWS["optuna_valid_end"])
].copy()
sim_test_df = all_lagged_ev[
    (all_lagged_ev["Date"] >= WINDOWS["sim_test_start"]) &
    (all_lagged_ev["Date"] <= WINDOWS["sim_test_end"])
].copy()

sim_model, sim_metrics, sim_pred_df = evaluate_model_on_period(
    tr_va_df, sim_test_df, best_fc, target_col, close_col, N_DAYS_BEST,
    best_cfg["model_name"], best_cfg["model_params"], best_cfg["pred_threshold"], RANDOM_STATE)

sim_summary_df = summarize_sim(sim_pred_df, target_col, N_DAYS_BEST)
take_profit_th = best_cfg["target_return_threshold"]
cash_sim_df = make_cash_simulation(
    sim_pred_df, close_col, target_col, N_DAYS_BEST,
    take_profit_threshold=take_profit_th,
    initial_cash=INITIAL_CASH, buy_ratio=BUY_RATIO, min_cash_ratio=MIN_CASH_RATIO)

print("[Simulation Test 요약]")
display_df(sim_summary_df.T.rename(columns={0:"value"}))
print()
print("pred 기준 action 분포:")
print(cash_sim_df["trade_action"].value_counts(dropna=False))
print()
print("what-if 기준 action 분포:")
print(cash_sim_df["trade_action_if"].value_counts(dropna=False))

final_asset    = cash_sim_df["asset"].iloc[-1]
final_asset_if = cash_sim_df["asset_if"].iloc[-1]
print(f"\n최종 자산 (pred 기준): {final_asset:,.0f}  ({(final_asset/INITIAL_CASH-1)*100:.1f}%)")
print(f"최종 자산 (what-if):   {final_asset_if:,.0f}  ({(final_asset_if/INITIAL_CASH-1)*100:.1f}%)")


[Simulation Test 요약]


,value
target_col,target_10d_up_4pct
n_days,10
eval_count,230
actual_1_count,96
pred_1_count,35
accuracy,0.56087
precision,0.428571
recall,0.15625
f1,0.229008
roc_auc,0.471082



pred 기준 action 분포:
trade_action
HOLD                    177
BUY                      27
SELL_EXPIRE              12
SELL_TAKE_PROFIT_BUY      6
SELL_TAKE_PROFIT          6
SELL_EXPIRE_BUY           2
Name: count, dtype: int64

what-if 기준 action 분포:
trade_action_if
HOLD                    111
BUY                      72
SELL_TAKE_PROFIT_BUY     24
SELL_TAKE_PROFIT         23
Name: count, dtype: int64

최종 자산 (pred 기준): 1,050,816  (5.1%)
최종 자산 (what-if):   1,303,746  (30.4%)


In [16]:
# ── 그래프 1: pred vs actual_target (0/1 비교) ─────────────────────────────
fig_pred_actual = go.Figure()

fig_pred_actual.add_trace(go.Scatter(
    x=sim_pred_df["Date"],
    y=sim_pred_df["pred"].astype(float) + 0.04,
    mode="lines+markers",
    name="pred (모델 신호)",
    line=dict(width=2, dash="dot"),
    marker=dict(size=7, symbol="x"),
))
fig_pred_actual.add_trace(go.Scatter(
    x=sim_pred_df["Date"],
    y=sim_pred_df[target_col].astype(float),
    mode="lines+markers",
    name="actual_target (실제)",
    line=dict(width=2),
    marker=dict(size=7, symbol="circle"),
))
fig_pred_actual.update_layout(
    title=f"{ETF_CODE} Simulation Test — pred vs actual_target",
    xaxis_title="Date", yaxis_title="0 / 1",
    hovermode="x unified", height=420,
    yaxis=dict(tickvals=[0, 1], ticktext=["0","1"], range=[-0.2, 1.4]),
)
fig_pred_actual.show()

# ── 그래프 2: 자산 추이 비교 (pred 전략 vs what-if 완벽예측) ──────────────
fig_cash_flow = go.Figure()

fig_cash_flow.add_trace(go.Scatter(
    x=cash_sim_df["Date"], y=cash_sim_df["asset"],
    mode="lines", name="자산 (pred 기준)",
    line=dict(width=2),
))
fig_cash_flow.add_trace(go.Scatter(
    x=cash_sim_df["Date"], y=cash_sim_df["asset_if"],
    mode="lines", name="자산 (what-if: 완벽예측)",
    line=dict(width=2, dash="dash"),
))
fig_cash_flow.add_hline(
    y=INITIAL_CASH, line_dash="dot", line_color="gray",
    annotation_text="초기자산", annotation_position="bottom right",
)
fig_cash_flow.update_layout(
    title=f"{ETF_CODE} Simulation Test — 자산 추이: pred 전략 vs what-if",
    xaxis_title="Date", yaxis_title="자산 (KRW)",
    hovermode="x unified", height=420,
)
fig_cash_flow.show()


In [12]:
# BASE_DATE 이전 전체 데이터로 재학습
final_train_df = all_lagged_ev[all_lagged_ev["Date"] <= WINDOWS["base_date"]].copy()

X_tr = final_train_df[best_fc].replace([np.inf,-np.inf], np.nan)
y_tr = final_train_df[target_col].astype(int)
valid_mask = X_tr.notnull().all(axis=1) & y_tr.notna()
X_tr = X_tr[valid_mask]; y_tr = y_tr[valid_mask]

pos = int((y_tr==1).sum()); neg = int((y_tr==0).sum())
final_model = make_classifier(best_cfg["model_name"], RANDOM_STATE,
                               best_cfg["model_params"], pos, neg)
if best_cfg["model_name"] == "gradient_boosting":
    sw = compute_sample_weight("balanced", y_tr)
    final_model.fit(X_tr, y_tr, sample_weight=sw)
else:
    final_model.fit(X_tr, y_tr)

# 최신 row 예측
latest_row = all_lagged_in.iloc[[-1]].copy()
latest_date = latest_row["Date"].values[0]
X_latest = latest_row[best_fc].replace([np.inf,-np.inf], np.nan)
pred_proba_latest = final_model.predict_proba(X_latest)[:,1][0]
pred_latest = int(pred_proba_latest >= best_cfg["pred_threshold"])

real_inference = {
    "as_of_date":       str(latest_date)[:10],
    "base_date":        BASE_DATE,
    "model":            best_cfg["model_name"],
    "n_days":           N_DAYS_BEST,
    "target_col":       target_col,
    "pred_proba":       float(pred_proba_latest),
    "pred_threshold":   best_cfg["pred_threshold"],
    "signal":           "BUY" if pred_latest==1 else "WAIT",
}
print(json.dumps(real_inference, ensure_ascii=False, indent=2))


{
  "as_of_date": "2026-06-12",
  "base_date": "2026-05-31",
  "model": "lightgbm",
  "n_days": 10,
  "target_col": "target_10d_up_4pct",
  "pred_proba": 0.1678794082441206,
  "pred_threshold": 0.4,
  "signal": "WAIT"
}


In [13]:
base_tag = f"{ETF_CODE}_{pd.to_datetime(BASE_DATE).strftime('%Y%m%d')}"

# 개별 실험 결과 CSV
sim_pred_df.to_csv(OUTPUT_DIR / f"sim_pred_{base_tag}.csv", index=False)
cash_sim_df.to_csv(OUTPUT_DIR / f"cash_sim_{base_tag}.csv", index=False)
best_artifacts["importance_df"].to_csv(OUTPUT_DIR / f"feature_importance_{base_tag}.csv", index=False)
stage1_result["trials_df"].to_csv(OUTPUT_DIR / f"stage1_trials_{base_tag}.csv", index=False)
stage2_result["trials_df"].to_csv(OUTPUT_DIR / f"stage2_trials_{base_tag}.csv", index=False)

with open(OUTPUT_DIR / f"best_config_{base_tag}.json", "w", encoding="utf-8") as f:
    json.dump(best_cfg, f, ensure_ascii=False, indent=2, default=str)
with open(OUTPUT_DIR / f"real_inference_{base_tag}.json", "w", encoding="utf-8") as f:
    json.dump(real_inference, f, ensure_ascii=False, indent=2, default=str)

fig_pred_actual.write_html(OUTPUT_DIR / f"plot_pred_vs_actual_{base_tag}.html")
fig_cash_flow.write_html(OUTPUT_DIR / f"plot_cash_flow_{base_tag}.html")

# ── 실험 비교 Summary CSV (누적) ──────────────────────────────────────────────
summary_path = Path("experiments_v3/experiment_summary.csv")
summary_path.parent.mkdir(parents=True, exist_ok=True)

s1row = sim_summary_df.iloc[0].to_dict()
summary_row = {
    "experiment_name":  EXPERIMENT_NAME,
    "run_date":         pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "etf":              ETF_CODE,
    "base_date":        BASE_DATE,
    "best_n_days":      best_cfg["n_days"],
    "best_threshold":   best_cfg["target_return_threshold"],
    "best_model":       best_cfg["model_name"],
    "best_top_n":       best_cfg["top_n"],
    "best_pred_threshold": best_cfg["pred_threshold"],
    "stage2_score":     best_cfg["score"],
    "sim_precision":    s1row.get("precision"),
    "sim_recall":       s1row.get("recall"),
    "sim_f1":           s1row.get("f1"),
    "sim_precision_lift": s1row.get("precision_lift"),
    "sim_precision_lift_recall": s1row.get("precision_lift_recall"),
    "sim_compound_return": s1row.get("strategy_compound_return"),
    "sim_avg_ret_buy":  s1row.get("avg_ret_when_buy"),
    "cash_sim_final_value": cash_sim_df["asset"].iloc[-1],
    "cash_sim_return_pct":  (cash_sim_df["asset"].iloc[-1] / INITIAL_CASH - 1) * 100,
    "real_signal":      real_inference["signal"],
    "real_pred_proba":  real_inference["pred_proba"],
    "s1_combos": len(N_DAYS_CANDIDATES) * len(TARGET_RETURN_THRESHOLD_CANDIDATES),
    "s1_seeds":  S1_N_SEEDS,
    "s2_n_trials": S2_N_TRIALS,
}

if summary_path.exists():
    exist_df = pd.read_csv(summary_path)
    new_df   = pd.concat([exist_df, pd.DataFrame([summary_row])], ignore_index=True)
else:
    new_df = pd.DataFrame([summary_row])

new_df.to_csv(summary_path, index=False)
print(f"\n결과 저장 완료: {OUTPUT_DIR.resolve()}")
print(f"Summary CSV: {summary_path.resolve()}")
display_df(new_df)



결과 저장 완료: /Users/jongheelee/Desktop/JH/01. Working/1. ETF_Prediction/etf_predict/experiments_v3/v3_baseline
Summary CSV: /Users/jongheelee/Desktop/JH/01. Working/1. ETF_Prediction/etf_predict/experiments_v3/experiment_summary.csv


,experiment_name,run_date,etf,base_date,best_n_days,best_threshold,best_model,best_top_n,best_pred_threshold,stage2_score,...,sim_compound_return,sim_avg_ret_buy,cash_sim_final_value,cash_sim_return_pct,real_signal,real_pred_proba,s1_n_trials,s2_n_trials,s1_combos,s1_seeds
0,v3_baseline,2026-06-14 13:45,SMH,2026-05-31,10,0.03,lightgbm,10,0.7,0.156585,...,17.029858,0.050986,1.081604e+06,8.160368,WAIT,0.503565,60.0,200,NaN,NaN
1,v3_baseline,2026-06-14 14:18,SMH,2026-05-31,10,0.04,lightgbm,5,0.4,0.092642,...,2.513178,0.038246,1.050816e+06,5.081579,WAIT,0.167879,NaN,200,12.0,3.0
